In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path

#
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Core.logging import flow_logger as logger
from Instruments.GX_271.dim import DIM
from Instruments.WPI_Syr_pump.Pump import AL1000
from Instruments.Runze_valves.Runze62Valve import Runze62Valve
from Ecosystems.reaction_segment_generation import RSG

In [2]:
# --- Instantiate serial object for AL1000 pump ----
ser = serial.Serial("COM2", 9600, timeout=0.5)
pump = AL1000(ser, device_address="@00", sleep_time=0.5)

pump.connect()

Device is connected


In [3]:
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Start logging (declare this run belongs to the experiment "Development") ---
logger.start_experiment("Development")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()
#DIM = DIM()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=156,
    y_offset=9,
    x_min=145,
    x_max=250,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=319,
    y_offset=39,
    x_min=275,
    x_max=370,
    y_min=2,
    y_max=292
) 

#tray.add_module(
    #slot=3,
    #name="dim",
   # module=dim,
  #  x_offset=0,     #These values are TBD
   # y_offset=0,
  #  x_min=0,
   # x_max=0,
    #y_min=0,
    #y_max=0
#) 

# Instantiate the RSG ecosystem with the connected Gilson and AL1000
rsg = RSG(gilson=g, pump=pump, syringe_diameter=4.606)

In [6]:
g.go_to_vial("rack1", 16)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(156.0), np.float64(276.40500000000003))

In [48]:
g.move_x(370)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


'Moved X to 370. Result: Move X: 2, Success'

In [20]:
print(g.current_y)

8.0


In [7]:
g.move_z(119)

'Moved Z to 119 at 125 mm/sec. Result: Move Z with Speed: 2, Success'

In [36]:
print(g.current_x)
print(g.current_y)

250.0
39.0


In [67]:
def run_prepickup_validation(
    rsg,
    analyte_volumes,
    start_vial_pos=17,
    n_replicates=2,
    prepickup_volume=20,
    airgap1_volume =20,
    withdraw_rate=0.2,   
    infuse_rate=0.5,      
):
    """
    Automated prepickup validation run.

    Logic:
    - Cold start creates full air gap
    - For each analyte volume:
        - Run all replicates sequentially
        - Wash between every replicate
        - Infuse into consecutive vials on rack1
    """

    print("=== Starting prepickup validation run ===")

    vial_pos = start_vial_pos
    airgap1_dispensed = airgap1_volume / 2

    # ---- Cold start only once ----
    print("Cold start: creating airgap1")
    rsg.take_air_gap(airgap1_volume, rate=withdraw_rate)

    for vol in analyte_volumes:
        print(f"\n=== Analyte volume: {vol} µL ===")

        for rep in range(1, n_replicates + 1):
            print(f"  Replicate {rep} → rack1 vial {vial_pos}")

            # 1) Regenerate air gap (skip ONLY for very first run)
            if not (vol == analyte_volumes[0] and rep == 1):
                rsg.take_air_gap(airgap1_dispensed, rate=withdraw_rate)

            # 2) Wash (internally uses its own fast/slow rates)
            rsg.wash_sequence()

            # 3) Prepickup
            rsg.prepickup(
                volume=prepickup_volume,
                rate=withdraw_rate,
            )

            # 4) Small air gap before analyte
            rsg.take_air_gap(5, rate=withdraw_rate)

            # 5) Pickup analyte
            rsg.pickup_from_vial(
                module_name="rack1",
                vial_pos=1,
                volume=vol,
                rate=withdraw_rate,
            )

            # 6) Dispense everything into dilution vial
            dispense_volume = (
                vol
                + 5
                + prepickup_volume
                + airgap1_dispensed
            )

            rsg.dispense_in_vial(
                module_name="rack1",
                vial_pos=vial_pos,
                volume=dispense_volume,
                rate=infuse_rate,
            )

            vial_pos += 1
            time.sleep(2)

    print("\n~~~ All done ~~~")



In [4]:
g.home()

All axes homed successfully and Z is within safe limits


In [8]:
print(g.current_x)
print(g.current_y)

0.0
0.0


In [7]:
g.move_xy(0,0)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


'Moved to X=0, Y=0. Result: Move XY: 2, Success'

In [75]:
analyte_volumes = [50, 40, 30, 20, 10]

run_prepickup_validation(
    rsg=rsg,
    analyte_volumes=analyte_volumes,
    start_vial_pos=2,
    n_replicates=3,
    prepickup_volume=20,
    withdraw_rate=0.2,
    infuse_rate=0.5,
)

# Note - the final state of this for-loop based automated run is working fluid + 10uL of the
# 20uL air gap left in the syringe. To get ready for further runs, run the two cells below this one.

=== Starting prepickup validation run ===
Cold start: creating airgap1
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.2 MM
Sent: @00*VOL20
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP

=== Analyte volume: 50 µL ===
  Replicate 1 → rack1 vial 2
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL100.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL105.0
Sent: @00*DIRINF
Sent: @00*RUN 1
S

In [ ]:
def run_factorial_validation(rsg, df, analyte_volumes, n_repeats=1, start_vial_pos=17):
    """
    Runs a DoE-based prepickup validation, mirroring manual automated logic.
    Each design row is paired with each analyte volume.

    Parameters:
    - rsg: RSG instance
    - df: DataFrame of factorial runs (from CSV)
    - analyte_volumes: list of analyte volumes to test for each run
    - n_repeats: number of repeats per volume
    - start_vial_pos: starting vial position on rack1
    """
    print("=== Starting factorial prepickup validation run ===")
    next_vial = start_vial_pos

    for idx, row in df.iterrows():
        for rep in range(1, n_repeats + 1):
            for vol in analyte_volumes:
                print(f"Running {row['design_id']}, repeat {rep}, analyte volume {vol} µL")

                # 1) Cold start / air gap 1 (only first run ever)
                if idx == 0 and rep == 1 and vol == analyte_volumes[0]:
                    rsg.take_air_gap(volume=row['airgap1'], rate=row['withdrawal'])

                # 2) Regenerate air gap before run (skip first)
                if not (idx == 0 and rep == 1 and vol == analyte_volumes[0]):
                    rsg.take_air_gap(10, rate=row['withdrawal'])

                # 3) Wash sequence
                rsg.wash_sequence()

                # 4) Prepickup
                rsg.prepickup(volume=row['prepickup'], rate=row['withdrawal'])

                # 5) Small air gap before analyte
                rsg.take_air_gap(5, rate=row['withdrawal'])

                # 6) Pickup analyte
                rsg.pickup_from_vial(module_name="rack1", vial_pos=1, volume=vol, rate=row['withdrawal'])

                # 7) Dispense everything into dilution vial
                dispense_volume = row['prepickup'] + 5 + row['airgap1']/2 + vol
                rsg.dispense_in_vial(
                    module_name="rack1",
                    vial_pos=next_vial,
                    volume=dispense_volume,
                    rate=row['infusion']
                )

                next_vial += 1
                time.sleep(2)

    print("\n~~~ All factorial runs complete ~~~")





In [ ]:
# Path to the CSV generated earlier - generate the design + save as CSV, then copy its path below
csv_path = Path("Design/outputs/factorial_runs_VB-1-6_2026-01-22.csv")

# Read into a DataFrame
df = pd.read_csv(csv_path)

# Optional: inspect first few rows
#df.head()


In [ ]:
analyte_volumes = [50, 40, 30, 20, 10]
n_repeats = 1     # Change as needed

run_factorial_validation(
    rsg=rsg,
    df=df,
    start_vial_pos=17,
    analyte_volumes=analyte_volumes,
    n_repeats=n_repeats
)

In [13]:
g.go_to_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(201.0), np.float64(106.0))

In [11]:
g.

Z below safe limit (110.00 < 130.00) — raising first.
All axes homed successfully and Z is within safe limits


In [59]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [47]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(201.0), np.float64(106.0), 20)

In [53]:
g.move_z(120)

'Moved Z to 120. Result: Move Z: 2, Success'

In [12]:
g.home()

All axes homed successfully and Z is within safe limits


In [65]:
rate = 0.25          # mL/min
volume = 750           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

pump.prepare(rate=rate, volume=volume, direction=direction)

Sent: @00*PHN1
Sent: @00*RAT C 0.25 MM
Sent: @00*VOL750
Sent: @00*DIRINF


In [66]:
pump.start()

Sent: @00*RUN 1


'00I'

In [63]:
pump.stop()

Sent: @00*STP


'00P'

In [58]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(201.0), np.float64(106.0), 20)

In [10]:
g.move_z(130)

'Moved Z to 130. Result: Move Z: 2, Success'

In [79]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [80]:
rsg.wash_sequence()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL100.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL105.0
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [73]:
rsg.prepickup()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [74]:
rsg.take_air_gap(5)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [19]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(36.0), np.float64(8.0))

In [6]:
g.move_z(60)

'Moved Z to 60. Result: No valid response received.'

In [75]:
rsg.pickup_from_vial("rack1", 1, 4)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL4
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [76]:
g.go_to_vial("rack1", 25)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(52.55), np.float64(159.5295))

In [77]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [78]:
rsg.dispense_in_vial("rack1", 25, 29)

#Note - analyte vol + 5 + 10 + 10 (final 10 is 10uL of the 40uL air gap, leaving 30uL,
# this 10uL is reintroduced before the wash sequence)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL29
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP
